# 04 - Chest Radiograph Exploration & Grad-CAM Saliency
## MedAgent: Multimodal Clinical Decision Support System

This notebook demonstrates:
1. Chest X-Ray radiograph loading and transforms
2. DenseNet-121 model inference
3. Gradient-weighted Class Activation Mapping (**Grad-CAM**)
4. Anatomical quadrant localization of findings


In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torch

from src.config.paths import CHECKPOINTS_DIR
from src.data.loaders.image_loader import XRAY_CLASSES, generate_synthetic_xray_image, get_xray_transforms
from src.evaluation.explainability import GradCAM, identify_anatomical_quadrants, overlay_cam_on_image
from src.models.vision.densenet import ChestXRayDenseNet121


### 1. Generating & Visualizing Radiological Findings


In [ ]:
fig, axes = plt.subplots(1, len(XRAY_CLASSES), figsize=(18, 4))

for idx, pathology in enumerate(XRAY_CLASSES):
    img = generate_synthetic_xray_image(pathology=pathology, image_size=224)
    axes[idx].imshow(img)
    axes[idx].set_title(pathology, fontsize=12, fontweight="bold")
    axes[idx].axis("off")

plt.suptitle("Synthesized Chest Radiographs across 5 Pathology Classes", fontsize=15)
plt.tight_layout()
plt.show()


### 2. DenseNet-121 Model Loading & Inference


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Inference Device: {device}")

model = ChestXRayDenseNet121(num_classes=5, pretrained=False).to(device)

ckpt_path = CHECKPOINTS_DIR / "xray_densenet_best.pth"
if ckpt_path.exists():
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    model.load_state_dict(ckpt["model_state_dict"])
    print("Loaded trained DenseNet-121 checkpoint!")
model.eval()

# Select Pneumonia test sample
sample_img = generate_synthetic_xray_image(pathology="Pneumonia", image_size=224)
transform = get_xray_transforms(image_size=224, is_training=False)
img_tensor = transform(sample_img).unsqueeze(0).to(device)

with torch.no_grad():
    logits = model(img_tensor)
    probs = torch.softmax(logits, dim=1).squeeze().cpu().numpy()

pred_idx = int(np.argmax(probs))
print(f"Predicted Class: {XRAY_CLASSES[pred_idx]} ({probs[pred_idx]*100:.2f}% confidence)")


### 3. Grad-CAM Saliency Map Generation & Anatomical ROI Localization


In [ ]:
grad_cam = GradCAM(model=model, target_layer=model.get_target_layer())
cam = grad_cam.generate_cam(img_tensor, target_class=pred_idx)
overlay = overlay_cam_on_image(sample_img, cam, alpha=0.5)

regions = identify_anatomical_quadrants(cam, threshold=0.5)
print(f"Salient Anatomical Regions Identified: {regions}")

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(sample_img)
axes[0].set_title("Input Chest X-Ray", fontsize=13)
axes[0].axis("off")

axes[1].imshow(cam, cmap="jet")
axes[1].set_title("Grad-CAM Activation Map", fontsize=13)
axes[1].axis("off")

axes[2].imshow(overlay)
axes[2].set_title(f"Explainability Overlay ({XRAY_CLASSES[pred_idx]})", fontsize=13)
axes[2].axis("off")

plt.tight_layout()
plt.show()
